## Config

In [1]:
import pickle, os
import pandas as pd, numpy as np
from collections import defaultdict

DATE       = '07262026'
CJ_DE = 'Important_genes/CJ_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026/CJ_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026.pkl'
AC_DE = 'Important_genes/AC_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026/AC_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026.pkl'
XT_DE = 'Important_genes/XT_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026/XT_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026.pkl'
DR_DE = 'Important_genes/DR_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026/DR_cleaned_sssubclass_v4_or_hypobkgd_sigupdate_07262026.pkl'
DE_PATH = {'CJ':CJ_DE,'AC':AC_DE,'XT':XT_DE,'DR':DR_DE}
OTO_PATH   = 'OTO_star_nothreshold_missing_le2_expressionthresh_07262026.tsv'
TF_PATH    = 'Fin_TF_06012026.csv'

SPECIES    = ['CJ', 'AC', 'XT', 'DR']   # non-mammal species: DE files + OTO columns

# --- conservation stringency knob -------------------------------------------------
# A TF is "conserved" in a group when it is DE in at least `required(group_size)`
# member species. Set MIN_SPECIES to one of:
#   'all'                -> DE in every member          (2/3/4 -> 2/3/4, strict)
#    N   (int)           -> DE in >= N members          (groups with < N members lack)
#   {2: 2, 3: 3, 4: 3}   -> explicit bar per group size (this gives 2/3/3: a
#                           4-species group needs only 3 of its 4 members)
# Shortcut for a "cap at K" rule: {n: min(n, K) for n in (2, 3, 4)}  (K=3 -> 2/3/3).
MIN_SPECIES = {2: 2, 3: 3, 4: 4}


def required(n_members):
    """How many member species a TF must be DE in for this group size."""
    if MIN_SPECIES == 'all':
        return n_members
    if isinstance(MIN_SPECIES, dict):
        return MIN_SPECIES[n_members]
    return int(MIN_SPECIES)

## 1. Load DE gene sets

One dict per species: `{subclass label -> set of DE gene symbols}`. Symbols are kept in native
casing (CJ upper, ac/xt/dr lower) so they line up with the OTO species columns.

In [2]:
de = {}
for S in SPECIES:
    spkl = DE_PATH[S]
    with open(spkl, 'rb') as fh:
        raw = pickle.load(fh)
    de[S] = {k: set(v) for k, v in raw.items()}
    print(f'{S}: {len(de[S])} subclasses, '
          f'{sum(len(v) for v in de[S].values())} DE gene calls')

CJ: 81 subclasses, 21208 DE gene calls
AC: 73 subclasses, 18319 DE gene calls
XT: 83 subclasses, 17626 DE gene calls
DR: 38 subclasses, 28554 DE gene calls


## 2. Shared subclasses

The member species are the `cj/ac/xt/dr` tokens in the label. A subclass is shared if it names
>=2 species and is present in each of those species' DE dict (asserted). Mouse-merged tokens
(`m066`) and Allen labels never match, so single-species and pass-through types drop out.

In [3]:
def participants(label):
    toks = set(label.split('_'))
    return [p.upper() for p in ['cj', 'ac', 'xt', 'dr'] if p in toks]


shared = {}
for S in SPECIES:
    for label in de[S]:
        pit = participants(label)
        if len(pit) >= 2:
            shared[label] = pit

for label, pit in shared.items():
    missing = [S for S in pit if label not in de[S]]
    assert not missing, f'{label}: named {pit} but absent from {missing}'

print(f'{len(shared)} shared subclasses')
for label, pit in sorted(shared.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    print(f'  {label:<16} {pit}')

21 shared subclasses
  cj_ac_xt_dr_1    ['CJ', 'AC', 'XT', 'DR']
  cj_ac_xt_dr_2    ['CJ', 'AC', 'XT', 'DR']
  cj_ac_xt_dr_3    ['CJ', 'AC', 'XT', 'DR']
  cj_ac_xt_dr_4    ['CJ', 'AC', 'XT', 'DR']
  cj_ac_xt_dr_5    ['CJ', 'AC', 'XT', 'DR']
  ac_xt_dr_1       ['AC', 'XT', 'DR']
  cj_ac_xt_1       ['CJ', 'AC', 'XT']
  cj_xt_dr_1       ['CJ', 'XT', 'DR']
  cj_xt_dr_2       ['CJ', 'XT', 'DR']
  ac_dr_1          ['AC', 'DR']
  ac_dr_2          ['AC', 'DR']
  ac_xt_1          ['AC', 'XT']
  ac_xt_2          ['AC', 'XT']
  ac_xt_3          ['AC', 'XT']
  ac_xt_4          ['AC', 'XT']
  cj_ac_1          ['CJ', 'AC']
  cj_ac_2          ['CJ', 'AC']
  cj_ac_3          ['CJ', 'AC']
  cj_ac_4          ['CJ', 'AC']
  cj_xt_1          ['CJ', 'XT']
  xt_dr_1          ['XT', 'DR']


## 3. Orthogroups and TF orthogroups

Orthogroups are identified by their **mouse (`MM`) symbol** — unique and non-null in this OTO
table, so it's a safe one-to-one id. `gene2og[S]` maps a native symbol to the orthogroup
(mouse-name) it sits in; `tf_ogs` are the orthogroups whose mouse name is in the TF list; and
`og_present[S]` records which orthogroups actually have a gene in species `S`. Since each `og`
id already *is* the mouse TF name, it's used directly as the TF label downstream.

In [4]:
# Orthogroups are keyed by their MOUSE (MM) symbol. MM is unique and non-null in this
# OTO table (verified), so the mouse name is a safe one-to-one id for each orthogroup.
oto = pd.read_csv(OTO_PATH, sep='\t', index_col='MM')

# native symbol -> set of orthogroup (mouse-name) ids it sits in, per species
gene2og = {S: defaultdict(set) for S in SPECIES}
for og, row in oto.iterrows():
    for S in SPECIES:
        g = row[S]
        if isinstance(g, str):
            gene2og[S][g].add(og)

# orthogroups that actually have a gene in species S
og_present = {S: set(oto.index[oto[S].notna()]) for S in SPECIES}

# TF orthogroups: mouse names in the TF list. The og id *is* the mouse TF name.
TF_set = set(pd.read_csv(TF_PATH)['0'].dropna().astype(str))
tf_ogs = set(oto.index) & TF_set

print(f'{len(TF_set)} TF symbols -> {len(tf_ogs)} TF orthogroups in OTO '
      f'(of {len(oto)} orthogroups total)')

1256 TF symbols -> 857 TF orthogroups in OTO (of 14485 orthogroups total)


## 4. DE TF orthogroups per (species, subclass)

For a member species, intersect its DE genes for the subclass with the TF orthogroups.

In [5]:
def de_tf_ogs(S, label):
    return {og
            for g in de[S].get(label, ())
            for og in gene2og[S].get(g, ())
            if og in tf_ogs}

## 5. Conserved-TF long table

One row per (subclass, TF orthogroup) that is DE in at least one member species. `n_required` is
the `MIN_SPECIES` bar for that group size and `conserved` is `n_DE >= n_required`. For the rest,
`present_not_DE` vs `ortholog_absent` says whether a miss is biological or just a missing
ortholog. Re-run this cell (and the ones below) after changing `MIN_SPECIES` in Config.

In [6]:
rows = []
for label, pit in sorted(shared.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    per = {S: de_tf_ogs(S, label) for S in pit}
    req = required(len(pit))
    for og in sorted(set().union(*per.values())):
        de_in   = [S for S in pit if og in per[S]]
        not_de  = [S for S in pit if og not in per[S] and og in og_present[S]]
        absent  = [S for S in pit if og not in og_present[S]]
        rows.append({
            'subclass'           : label,
            'n_species_in_group' : len(pit),
            'group_species'      : ','.join(pit),
            'TF'                 : og,          # og id is the mouse TF name
            'n_DE'               : len(de_in),
            'n_required'         : req,
            'DE_in'              : ','.join(de_in),
            'present_not_DE'     : ','.join(not_de),
            'ortholog_absent'    : ','.join(absent),
            'conserved'          : len(de_in) >= req,
        })

tf_long = pd.DataFrame(rows)
print(f'{tf_long.shape[0]} (subclass, TF) rows | '
      f'{int(tf_long["conserved"].sum())} conserved at MIN_SPECIES={MIN_SPECIES!r}')
tf_long.head(20)

1216 (subclass, TF) rows | 130 conserved at MIN_SPECIES={2: 2, 3: 3, 4: 4}


,subclass,n_species_in_group,group_species,TF,n_DE,n_required,DE_in,present_not_DE,ortholog_absent,conserved
0,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Arntl,1,4,DR,"CJ,AC,XT",,False
1,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Bach1,1,4,DR,"CJ,AC,XT",,False
2,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Bach2,1,4,DR,"CJ,AC,XT",,False
3,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Barhl1,3,4,"CJ,AC,XT",DR,,False
4,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Barhl2,2,4,"CJ,AC","XT,DR",,False
5,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Bcl11a,1,4,DR,"CJ,AC,XT",,False
6,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Bcl11b,3,4,"CJ,XT,DR",AC,,False
7,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Bhlhe22,1,4,DR,"AC,XT",CJ,False
8,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Bhlhe23,1,4,DR,"CJ,AC,XT",,False
9,cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",Bhlhe40,1,4,DR,"CJ,AC,XT",,False


## 6. Per-subclass summary

Every shared subclass appears here (including those with zero DE TFs), sorted so the widest
groups and the most TF-poor come first.

Each entry in `conserved_TFs` is tagged with any member species the TF is **not** DE in — which
happens under a relaxed `MIN_SPECIES` (e.g. 2/3/3, where a 4-species group can be conserved on 3
of 4). `Lhx2(-DR)` means conserved but not DE in zebrafish; a trailing `*` (e.g. `Nfib(-DR*)`)
means DR lacks the ortholog entirely, so it could never have been DE. Untagged names are DE in
every member.

In [8]:
meta = pd.DataFrame(
    [{'subclass': l, 'n_species_in_group': len(p), 'group_species': ','.join(p)}
     for l, p in shared.items()]).set_index('subclass')

cons = tf_long[tf_long['conserved']].copy()


def _tf_label(row):
    """TF name, tagged with any member species it is NOT DE in (relaxed threshold).
    A trailing '*' marks species where the ortholog is absent (can't be DE) rather
    than merely not called DE."""
    absent = [s for s in row['ortholog_absent'].split(',') if s]
    not_de = [s for s in row['present_not_DE'].split(',') if s]
    missing = [f'{s}*' for s in absent] + not_de
    return row['TF'] if not missing else f"{row['TF']}(-{'/'.join(missing)})"


cons['tf_label'] = cons.apply(_tf_label, axis=1)

n_union = tf_long.groupby('subclass')['TF'].nunique().rename('n_TF_union')
n_cons  = cons.groupby('subclass')['TF'].nunique().rename('n_TF_conserved')
names   = (cons.dropna(subset=['TF']).groupby('subclass')['tf_label']
               .apply(lambda s: ', '.join(sorted(s.unique())))
               .rename('conserved_TFs'))

per_subclass = meta.join(n_union).join(n_cons).join(names)
per_subclass['n_TF_union']     = per_subclass['n_TF_union'].fillna(0).astype(int)
per_subclass['n_TF_conserved'] = per_subclass['n_TF_conserved'].fillna(0).astype(int)
per_subclass['conserved_TFs']  = per_subclass['conserved_TFs'].fillna('')
per_subclass = per_subclass.sort_values(['n_species_in_group', 'n_TF_conserved'],
                                        ascending=[False, True])

# conserved_TFs tags: 'Lhx2(-DR)' = conserved but not DE in DR; '*' = ortholog absent there.
pd.set_option('display.max_colwidth', 200)
per_subclass

,n_species_in_group,group_species,n_TF_union,n_TF_conserved,conserved_TFs
subclass,,,,,
cj_ac_xt_dr_5,4,"CJ,AC,XT,DR",60,0,
cj_ac_xt_dr_2,4,"CJ,AC,XT,DR",76,1,Mef2c
cj_ac_xt_dr_3,4,"CJ,AC,XT,DR",55,2,"Lhx2, Lhx9"
cj_ac_xt_dr_4,4,"CJ,AC,XT,DR",66,2,"Gata3, Tal1"
cj_ac_xt_dr_1,4,"CJ,AC,XT,DR",159,6,"Ebf1, Ebf2, Ebf3, Irx1, Irx2, Lef1"
ac_xt_dr_1,3,"AC,XT,DR",27,0,
cj_ac_xt_1,3,"CJ,AC,XT",43,4,"Ebf1, Pitx2, Sim1, Uncx"
cj_xt_dr_2,3,"CJ,XT,DR",49,4,"Prdm16, Zic1, Zic2, Zic4"
cj_xt_dr_1,3,"CJ,XT,DR",145,10,"Dach1, Dach2, Lef1, Nr2e1, Prdm13, Prox1, Sox1, Sp5, St18, Zfp536"


In [9]:
per_subclass.index

Index(['cj_ac_xt_dr_5', 'cj_ac_xt_dr_2', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4',
       'cj_ac_xt_dr_1', 'ac_xt_dr_1', 'cj_ac_xt_1', 'cj_xt_dr_2', 'cj_xt_dr_1',
       'ac_dr_2', 'xt_dr_1', 'cj_xt_1', 'ac_xt_3', 'cj_ac_1', 'ac_xt_4',
       'ac_xt_2', 'cj_ac_2', 'ac_xt_1', 'ac_dr_1', 'cj_ac_3', 'cj_ac_4'],
      dtype='object', name='subclass')

## 7. Save

In [10]:
if MIN_SPECIES == 'all':
    tag = 'all'
elif isinstance(MIN_SPECIES, dict):
    tag = 'req' + ''.join(str(MIN_SPECIES[k]) for k in sorted(MIN_SPECIES))  # e.g. req233
else:
    tag = f'min{int(MIN_SPECIES)}'

cons.sort_values(['n_species_in_group', 'subclass', 'TF'],
                 ascending=[False, True, True]).to_csv(
    f'nonmam_conserved_DE_TFs_{tag}_{DATE}.csv', index=False)
tf_long.to_csv(f'nonmam_subclass_TF_long_{tag}_{DATE}.csv', index=False)
per_subclass.to_csv(f'nonmam_subclass_TF_summary_{tag}_{DATE}.csv')
print('wrote:')
for f in (f'nonmam_conserved_DE_TFs_{tag}_{DATE}.csv',
          f'nonmam_subclass_TF_long_{tag}_{DATE}.csv',
          f'nonmam_subclass_TF_summary_{tag}_{DATE}.csv'):
    print('  ', f)

wrote:
   nonmam_conserved_DE_TFs_req234_07262026.csv
   nonmam_subclass_TF_long_req234_07262026.csv
   nonmam_subclass_TF_summary_req234_07262026.csv


In [12]:
for item in per_subclass.index:
    if per_subclass.loc[item,'n_TF_conserved'] < 2:
        print(item)

cj_ac_xt_dr_5
cj_ac_xt_dr_2
ac_xt_dr_1
ac_dr_2
xt_dr_1
cj_xt_1


## 8. Revert low-support mappings

Dissolve the shared subclasses that carry too few conserved DE TFs back to their original
per-species v4 labels. Reads the consensus column (`ss_subclass_nounlabeled_nmm_v4_nn`) and
writes a **cleaned** copy to a new column `ss_subclass_nounlabeled_nmm_cl_v4_nn`: for every cell
whose consensus label is in `REVERT_GROUPS`, the label is restored from
`ss_subclass_v4_nounlabeled_nn` (the node-level v4 label, e.g. `dr_5`); all other cells keep their
consensus label. The original consensus column is left untouched.

`REVERT_GROUPS` defaults to the six groups with **< 2 conserved TFs under the strict
`MIN_SPECIES='all'` criterion** — pinned explicitly rather than recomputed off the current
`MIN_SPECIES` (which would pick a different set).

The commit cell re-saves the four SAM h5ads (adding the new column) and is not reversible — the
first cell is a dry run; run the commit cell only once its report looks right.

In [11]:
import gc
from samalg import SAM

REVERT_LEVEL    = 'ss_subclass_v4_nounlabeled_nn'         # original per-species v4 node labels
CONSENSUS_LEVEL = 'ss_subclass_nounlabeled_nmm_v4_nn'     # consensus column from the mapping notebook (source, left intact)
OUTPUT_LEVEL    = 'ss_subclass_nounlabeled_nmm_cl_v4_nn'  # cleaned consensus written here (reverts applied)

SAM_DIR = 'Active_SAM_joined/'
H5AD = {
    'CJ': SAM_DIR + 'SAM_CJ_joined_v2_cleaned_03122025.h5ad',
    'AC': SAM_DIR + 'SAM_AC_ncbi_soupx_cleaned_03122025.h5ad',
    'XT': SAM_DIR + 'SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad',
    'DR': SAM_DIR + 'SAM_DR_ncbi_joined_cleaned_07172026.h5ad',
}

# Groups to dissolve back to their v4 labels. Pinned explicitly: these are the shared
# subclasses with < 2 conserved TFs under the STRICT MIN_SPECIES='all' criterion.
REVERT_GROUPS = ['cj_ac_xt_dr_5', 'cj_ac_xt_dr_2', 'ac_xt_dr_1', 'ac_dr_2', 'cj_xt_1', 'xt_dr_1']

bad = [g for g in REVERT_GROUPS if g not in shared]
assert not bad, f'not shared subclasses: {bad}'


def revert_h5ad(S, fn, dry_run=True):
    sam = SAM()
    sam.load_data(fn)
    revert_obs_names = sam.adata.obs_names[sam.adata.obs[CONSENSUS_LEVEL].isin(REVERT_GROUPS)]
    obs = sam.adata.obs
    for col in (REVERT_LEVEL, CONSENSUS_LEVEL):
        if col not in obs.columns:
            raise KeyError(f'{S}: {col} missing from {fn}')
            
    fin_labels = []
    for item in sam.adata.obs_names:
        if item in revert_obs_names:
            fin_labels.append(sam.adata.obs.loc[item,REVERT_LEVEL])
        else:
            fin_labels.append(sam.adata.obs.loc[item,CONSENSUS_LEVEL])

    hit = obs.loc[revert_obs_names, CONSENSUS_LEVEL].value_counts().to_dict()
    print(f'{S}: {len(revert_obs_names)} cells reverted  {hit}')
    print(f'    {OUTPUT_LEVEL}: {obs[CONSENSUS_LEVEL].nunique()} -> {pd.Series(fin_labels).nunique()} distinct labels')

    if dry_run:
        print('    dry run — nothing written')
        return sam
    else:
        sam.adata.obs[OUTPUT_LEVEL] = fin_labels
        sam.save_anndata(fn)
        print(f'    saved {OUTPUT_LEVEL} into {fn}')
        return sam


# Dry run: report only, writes nothing.
for S in SPECIES:
    revert_h5ad(S, H5AD[S], dry_run=True)
    gc.collect()

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CJ: 1171 cells reverted  {'cj_xt_1': 504, 'cj_ac_xt_dr_5': 348, 'cj_ac_xt_dr_2': 319, '004 L6 IT CTX Glut': 0, '325 CHOR NN': 0, '231 IPN-LDT Vsx2 Nkx6-1 Glut': 0, '235 PG-TRN-LRN Fat2 Glut': 0, '277 DTN-LDT-IPN Otp Pax3 Gaba': 0, '321 Astroependymal NN': 0, '322 Tanycyte NN': 0, '323 Ependymal NN': 0, '326 OPC NN': 0, '213 SCsg Gabrr2 Gaba': 0, '327 Oligo NN': 0, '329 ABC NN': 0, '330 VLMC NN': 0, '333 Endo NN': 0, '339 Astrocyte-like NN': 0, '215 SNc-VTA-RAmb Foxa1 Dopa': 0, '211 SC Tnnt1 Gli3 Gaba': 0, '212 SCs Lef1 Gli3 Gaba': 0, 'Unlabeled': 0, '209 SCs Pax7 Nfia Gaba': 0, '208 SC Lef1 Otx2 Gaba': 0, '207 SCs Dmbx1 Gaba': 0, '205 SC-PAG Lef1 Emx2 Gaba': 0, '203 LGv-SPFp-SPFm Nkx2-2 Tcf7l2 Gaba': 0, '202 PRT Tcf7l2 Gaba': 0, '190 ND-INC Foxd2 Glut': 0, '168 SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut': 0, '164 APN C1ql4 Glut': 0, '159 IF-RL-CLI-PAG Foxa1 Glut': 0, '153 MG-POL-SGN Nts Glut': 0, '152 RE-Xi Nox4 Glut': 0, '340 Macrophage NN': 0, 'cj_14': 0, 'cj_13': 0, '006 L4/5 IT CTX Glut': 

In [13]:
# COMMIT — overwrites the four SAM h5ads in place. Not reversible.
# Run only after the dry run above looks correct.
for S in SPECIES:
    revert_h5ad(S, H5AD[S], dry_run=False)
    gc.collect()

CJ: 1171 cells reverted  {'cj_xt_1': 504, 'cj_ac_xt_dr_5': 348, 'cj_ac_xt_dr_2': 319, '004 L6 IT CTX Glut': 0, '325 CHOR NN': 0, '231 IPN-LDT Vsx2 Nkx6-1 Glut': 0, '235 PG-TRN-LRN Fat2 Glut': 0, '277 DTN-LDT-IPN Otp Pax3 Gaba': 0, '321 Astroependymal NN': 0, '322 Tanycyte NN': 0, '323 Ependymal NN': 0, '326 OPC NN': 0, '213 SCsg Gabrr2 Gaba': 0, '327 Oligo NN': 0, '329 ABC NN': 0, '330 VLMC NN': 0, '333 Endo NN': 0, '339 Astrocyte-like NN': 0, '215 SNc-VTA-RAmb Foxa1 Dopa': 0, '211 SC Tnnt1 Gli3 Gaba': 0, '212 SCs Lef1 Gli3 Gaba': 0, 'Unlabeled': 0, '209 SCs Pax7 Nfia Gaba': 0, '208 SC Lef1 Otx2 Gaba': 0, '207 SCs Dmbx1 Gaba': 0, '205 SC-PAG Lef1 Emx2 Gaba': 0, '203 LGv-SPFp-SPFm Nkx2-2 Tcf7l2 Gaba': 0, '202 PRT Tcf7l2 Gaba': 0, '190 ND-INC Foxd2 Glut': 0, '168 SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut': 0, '164 APN C1ql4 Glut': 0, '159 IF-RL-CLI-PAG Foxa1 Glut': 0, '153 MG-POL-SGN Nts Glut': 0, '152 RE-Xi Nox4 Glut': 0, '340 Macrophage NN': 0, 'cj_14': 0, 'cj_13': 0, '006 L4/5 IT CTX Glut': 